# EN3150 Assignment 03 - Random Guess
## Resource-Constrained CNN for Edge Image Classification on DeFungi

Group members:

| Member | Index | Main contribution |
|---|---|---|
| Rifnaz.K.R.M | 230550P | Dataset Preparation + Model A |
| Panuharan.S | 230462X | Model B + Efficiency Analysis |
| Peranavan.K | 230474K | Optimizer Comparison + Training/Evaluation |
| Lavanathan.J | 230371R | Pretrained Models + Final Comparison |

This notebook is self-contained for review. It defines the dataset handling and model architectures inline, then loads the verified experiment outputs saved in this repository. Expensive full training is not run automatically.

## 1. Environment and Reproducibility

The project was verified with Python 3.12.3 and TensorFlow/Keras. The fixed random seed used by the repository is 42.

In [ ]:
from pathlib import Path
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf

SEED = 42
IMAGE_SIZE = (64, 64)
CHANNELS = 3
NUM_CLASSES = 5
BATCH_SIZE = 32
CLASS_NAMES = ["H1", "H2", "H3", "H5", "H6"]
PROJECT_ROOT = Path.cwd()
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw" / "defungi"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
REPORT_DIR = PROJECT_ROOT / "report"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__)
print("Project root", PROJECT_ROOT)

## 2. Dataset

Dataset: DeFungi, UCI Machine Learning Repository. Official page: https://archive.ics.uci.edu/dataset/773/defungi

Raw images are intentionally not included in the public repository. Place the extracted dataset at `data/raw/defungi/` with subfolders `H1`, `H2`, `H3`, `H5`, and `H6`.

In [ ]:
dataset_summary = pd.read_csv(RESULTS_DIR / "tables" / "dataset_summary.csv")
dataset_summary

## 3. Canonical Split Verification

The canonical split is deterministic and stratified. The saved split metadata is reused for all models.

In [ ]:
def load_split(name):
    path = PROJECT_ROOT / "data" / "splits" / f"{name}.csv"
    df = pd.read_csv(path)
    id_col = "path" if "path" in df.columns else "relative_path"
    return df, id_col

split_frames = {}
path_sets = {}
for split in ["train", "validation", "test"]:
    df, id_col = load_split(split)
    split_frames[split] = df
    path_sets[split] = set(df[id_col])
    print(split, len(df), df["class_name"].value_counts().sort_index().to_dict())

assert not path_sets["train"] & path_sets["validation"]
assert not path_sets["train"] & path_sets["test"]
assert not path_sets["validation"] & path_sets["test"]
assert sum(len(df) for df in split_frames.values()) == 9114
print("No split overlap; total coverage = 9114")

## 4. Preprocessing and Data Loading

Custom models use 64 x 64 RGB input with pixel values scaled to [0, 1]. Validation and test loading are deterministic. The code below is self-contained and does not import project modules.

In [ ]:
def make_custom_dataset_from_split(split_name, batch_size=BATCH_SIZE, shuffle=False):
    df, id_col = load_split(split_name)
    paths = [str(PROJECT_ROOT / p) for p in df[id_col]]
    labels = df["label"].astype("int32").to_numpy()

    def load_image(path, label):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
        image = tf.image.resize(image, IMAGE_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

print("Dataset loader defined. It requires raw images only when training/evaluating from pixels.")

## 5. Model A - Standard CNN

Model A uses standard Conv2D layers, ReLU activation, max pooling, global average pooling, and a five-class softmax head.

In [ ]:
def build_model_a(input_shape=(64, 64, 3), num_classes=5):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="model_a_standard_cnn")

model_a = build_model_a()
print("Model A parameters:", model_a.count_params())
assert model_a.output_shape[-1] == 5

In [ ]:
pd.read_csv(RESULTS_DIR / "tables" / "model_a_architecture.csv")

## 6. Model B - Depthwise Separable CNN

Model B uses Keras `SeparableConv2D` layers and is constrained to fewer than 100,000 trainable parameters.

In [ ]:
def build_model_b(input_shape=(64, 64, 3), num_classes=5):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.SeparableConv2D(32, 3, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.SeparableConv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.SeparableConv2D(96, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(48, activation="relu")(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs, name="model_b_lightweight_sepconv")

model_b = build_model_b()
print("Model B parameters:", model_b.count_params())
assert model_b.output_shape[-1] == 5
assert model_b.count_params() < 100_000

In [ ]:
pd.read_csv(RESULTS_DIR / "tables" / "model_b_architecture.csv")

## 7. Parameter Calculations

Standard convolution with bias:

`P_standard = (Kh * Kw * Cin + 1) * Cout`

Depthwise-separable convolution with Keras bias:

`P_sep = Kh * Kw * Cin + Cin * Cout + Cout`

ReLU was selected because it is hardware-friendly: it mainly uses a simple max operation and avoids expensive exponentials.

In [ ]:
pd.read_csv(RESULTS_DIR / "tables" / "depthwise_parameter_comparison.csv")

## 8. Optimizer Comparison

The optimizer pilot compared Adam, standard SGD, and SGD with momentum on Model B using the same canonical split. Adam was selected by lowest final validation loss.

In [ ]:
optimizer_comparison = pd.read_csv(RESULTS_DIR / "tables" / "optimizer_comparison.csv")
optimizer_comparison.groupby("optimizer").tail(1)[["optimizer", "learning_rate", "momentum", "validation_loss", "validation_accuracy"]]

## 9. Training Utilities

The repository training scripts record history, epoch times, metrics, classification reports, confusion matrices, and test predictions. To avoid expensive reruns, this notebook loads the saved verified histories.

In [ ]:
def load_history(model_key):
    return pd.read_csv(RESULTS_DIR / "raw" / model_key / "history.csv")

def load_metrics(model_key):
    with open(RESULTS_DIR / "raw" / model_key / "metrics.json", "r", encoding="utf-8") as f:
        return json.load(f)

for key in ["model_a", "model_b", "mobilenetv2", "efficientnetb0"]:
    hist = load_history(key)
    print(key, "epochs recorded:", len(hist))

## 10. Custom Model Results

In [ ]:
custom_models = pd.read_csv(RESULTS_DIR / "tables" / "custom_models.csv")
custom_models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for key, title in [("model_a", "Model A"), ("model_b", "Model B")]:
    hist = load_history(key)
    axes[0].plot(hist["epoch"], hist["loss"], label=f"{title} train")
    axes[0].plot(hist["epoch"], hist["val_loss"], linestyle="--", label=f"{title} val")
    axes[1].plot(hist["epoch"], hist["accuracy"], label=f"{title} train")
    axes[1].plot(hist["epoch"], hist["val_accuracy"], linestyle="--", label=f"{title} val")
axes[0].set_title("Loss")
axes[1].set_title("Accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 11. Confusion Matrices and Error Analysis

The most common custom-model confusion is H2 predicted as H1. H3 is also comparatively difficult for the custom models. EfficientNet-B0 improves recall for H3, H5, and H6 but still confuses many H2 images with H1.

In [ ]:
for key in ["model_a", "model_b", "mobilenetv2", "efficientnetb0"]:
    print("\n", key)
    print(pd.read_csv(RESULTS_DIR / "raw" / key / "classification_report.csv", index_col=0).round(4))

## 12. MobileNetV2 and EfficientNet-B0 Transfer Learning

Both pretrained models used ImageNet weights, 64 x 64 input, a five-class DeFungi head, three frozen-head epochs at learning rate 0.001, and eight fine-tuning epochs at learning rate 0.0001. Validation loss selected the saved checkpoint.

In [ ]:
pd.read_csv(RESULTS_DIR / "tables" / "pretrained_transfer_models.csv")

## 13. Profiling and Resource Evidence

MAC estimates and CPU batch-1 latency were measured using saved models and a transparent Conv/Dense counting method. Disk I/O and decoding are excluded from latency measurements.

In [ ]:
print("Custom latency")
display(pd.read_csv(RESULTS_DIR / "tables" / "custom_latency.csv"))
print("Pretrained latency")
display(pd.read_csv(RESULTS_DIR / "tables" / "pretrained_transfer_latency.csv"))

## 14. Final Comparison

In [ ]:
final_comparison = pd.read_csv(RESULTS_DIR / "tables" / "final_comparison.csv")
final_comparison

In [ ]:
ax = final_comparison.plot(x="model", y="test_accuracy", kind="bar", legend=False, figsize=(7,4))
ax.set_ylim(0, 0.85)
ax.set_ylabel("Test accuracy")
ax.set_title("Final accuracy comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 15. Conclusions

- Model A is the stronger custom baseline: 0.6959 test accuracy with 101,829 parameters.
- Model B satisfies the resource constraint: 14,272 parameters and 0.6484 test accuracy.
- EfficientNet-B0 gives the best accuracy: 0.7668 test accuracy.
- Model B has the smallest serialized size, lowest MAC estimate, and lowest measured CPU latency.
- Parameter count alone does not prove lower latency or lower energy; measured latency and deployment hardware matter.

## 16. Reproduction Instructions

1. Install dependencies with `pip install -r requirements.txt`.
2. Download DeFungi from UCI and place it at `data/raw/defungi/`.
3. Recreate/verify splits with `python -m src.data_prepare --data-dir data/raw/defungi`.
4. Run verification with `python -m src.verify_imports`, `python -m src.verify_environment`, and `python -m src.verify_pipeline`.
5. Custom training scripts: `python -m src.train_custom --model model_a --epochs 20` and `python -m src.train_custom --model model_b --epochs 20`.
6. Transfer-learning script: `python -m src.session3_transfer train --models mobilenetv2 efficientnetb0 --head-epochs 3 --fine-tune-epochs 8 --batch-size 32 --image-size 64 64`.
7. Final tables: `python -m src.session3_transfer finalize --warmup-runs 20 --timed-runs 200`.
8. Report compile: from `report/`, run `latexmk -pdf -interaction=nonstopmode -halt-on-error main.tex`.